In [ ]:
## Implementing simple chatbot using langgraph
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from typing import Annotated
# REDUCERS
from langgraph.graph.message import add_messages


In [ ]:
class State(TypedDict):
    messages:Annotated[list,add_messages]
    

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.7-flash",
    temperature=1.0, 
    max_tokens=None,
    timeout=None,
    max_retries=2,
)
model.invoke("Hello how can i assist you today? ")


In [ ]:
from langchain_groq import ChatGroq
llm_groq = ChatGroq(model = "meta-llama/llama-prompt-guard-2-86m")
llm_groq.invoke("Hi my name is divyanshu and i like to play cricket outside")

In [ ]:
## We will start with creating nodes
def superbot(state:State):
    return {"messages":[llm_groq.invoke(state["messages"])]}


In [ ]:
graph = StateGraph(State)
graph.add_node("SuperBot",superbot)
graph.add_edge(START,"SuperBot")
graph.add_edge("SuperBot",END)
graphbuilder = graph.compile()

In [ ]:
from IPython.display import Image,display 
display(Image(graphbuilder.get_graph().draw_mermaid_png()))

In [ ]:
# Invokation --> 
graphbuilder.invoke({"messages":"Hi,my name is Divyanshu and I like cricket"})


In [ ]:
# Streaming the responses
graphbuilder.stream({"messages":"Hello my name is divyanshu"})
for event in graphbuilder.stream({"messages":"Hello my name is divyanshu"}):
    print(event)